In [ ]:
def _coerce_severity_any(val) -> int:
    """Convert whatever came back (int/str/word/None) into an int in [1..5]."""
    # Already an int-like
    try:
        # handle numpy integer too without importing numpy explicitly
        if isinstance(val, bool):
            # protect against True/False turning into 1/0
            pass
        elif isinstance(val, int):
            n = val
            return 1 if n < 1 else 5 if n > 5 else n
    except Exception:
        pass

    # Strings like "3", " 4 ", or words
    s = ("" if val is None else str(val)).strip().lower()
    if s.isdigit():
        n = int(s)
        return 1 if n < 1 else 5 if n > 5 else n
    return SEVERITY_MAP.get(s, 1)

def _coerce_and_validate_entity(d: dict) -> dict:
    """
    Expect either:
      - raw extracted dict like {"note": str, "labels": {...}}
      - or final entity dict like {"note": ..., "condition": ..., "age_range": ..., "severity_grade": ..., "embedding": ...}
    Returns a new dict with guaranteed-correct types for Milvus.
    """
    # Detect shape and normalize to flat fields
    if "labels" in d:
        note = d.get("note") or d.get("text")
        labels = d["labels"]
        condition = str(labels.get("condition", "")).lower()
        age_range = str(labels.get("age_range", ""))
        severity = _coerce_severity_any(labels.get("severity_grade"))
        flat = {
            "note": note,
            "condition": condition if condition in CONDITIONS else "unknown",
            "age_range": age_range if age_range in AGE_RANGES else "36-55",
            "severity_grade": severity,
        }
    else:
        # Already flat; just coerce/check
        flat = dict(d)
        flat["condition"] = str(flat.get("condition", "")).lower()
        if flat["condition"] not in CONDITIONS:
            flat["condition"] = "unknown"
        ar = str(flat.get("age_range", ""))
        flat["age_range"] = ar if ar in AGE_RANGES else "36-55"
        flat["severity_grade"] = _coerce_severity_any(flat.get("severity_grade"))

    # Final assertions for fast fail
    assert isinstance(flat["severity_grade"], int), f"severity_grade not int: {flat['severity_grade']} ({type(flat['severity_grade'])})"
    assert 1 <= flat["severity_grade"] <= 5, f"severity out of range: {flat['severity_grade']}"
    return flat

def validate_and_fix_for_milvus(rows: list[dict]) -> list[dict]:
    """
    Runs coercion + validation on each row; prints offenders before fixing.
    Returns a NEW list of dicts ready for Milvus insert (correct types).
    """
    fixed = []
    offenders = []
    for idx, d in enumerate(rows):
        # Peek at current sev value for debugging
        sev = d["labels"]["severity_grade"] if "labels" in d else d.get("severity_grade")
        if not (isinstance(sev, int) and 1 <= sev <= 5):
            offenders.append((idx, sev, type(sev)))
        try:
            fixed.append(_coerce_and_validate_entity(d))
        except AssertionError as e:
            # Include row snippet for quick diagnosis
            snippet = d.get("note") or d.get("text") or str(d)
            raise AssertionError(f"Row {idx} failed validation: {e}\nRow snippet: {snippet[:140]}") from e

    if offenders:
        print("⚠️  Severity offenders detected (coercing to int):")
        for idx, sev, typ in offenders[:25]:  # cap printout
            print(f"  - row {idx}: severity_grade={repr(sev)} (type={typ})")
        if len(offenders) > 25:
            print(f"  ...and {len(offenders)-25} more")

    return fixed

In [ ]:
labeled_notes = extract_labels(NOTES)

notes: list = [ln["note"] for ln in labeled_notes]
embeddings: list[list[float]] = embed_notes(notes)
entities: list[dict] = []

for row, emb in zip(labeled_notes, embeddings):
    entities.append({
        "note": row["note"],
        "condition": row["labels"]["condition"],
        "age_range": row["labels"]["age_range"],
        "severity_grade": row["labels"]["severity_grade"],
        "embedding": emb,
    })

entities = validate_and_fix_for_milvus(entities)